# AMBER simulation setup for basic protein MD

Heavily based on the AMBER Tutorial 1 (Section 5), Tutorial 7, and the BioExcel biobb workflow.

The dashboard provides you with the molecule as `input.pdb`.
First, give it a nice name, specify the main simulation length, and adjust equilibration steps if necessary.

In [ ]:
name = "protein"  # give me a better name

nanoseconds = 0.05  # just for fun
nsteps = int(nanoseconds * 500000)  # assumes 2 fs timestep

nmin = 5000          # minimization cycles per stage
nheat = 25000        # heating steps (50 ps)
nnvt = 50000         # NVT equilibration steps (100 ps)
nnpt = 125000        # NPT equilibration steps (250 ps)
temp0 = 300.0        # target temperature (K)
salt_mM = 150.0      # salt concentration
buffer = 12.0        # solvation buffer (angstroms)

In [ ]:
import amber_wrapper as amb
import nglview as nv
import mdtraj as md
import numpy as np
import matplotlib.pyplot as plt

## Look at the input

Tune NGLView parameters if needed, inspect the input visually

In [ ]:
nv.show_file("input.pdb")

## Initial setup

### Prepare PDB

Run `pdb4amber` to clean the input: rename residues, handle disulfides, remove CONECT records.

In [ ]:
amb.pdb4amber(input="input.pdb", output=f"{name}_clean.pdb", reduce=True)

### Build topology and solvate with tleap

Generate force field, neutralize, solvate, and add salt.

In [ ]:
with open(f"{name}.leap.in", "w") as leap:
    leap.write(f"""\
source leaprc.protein.ff19SB
source leaprc.water.opc
protein = loadpdb {name}_clean.pdb
check protein
saveamberparm protein {name}_gas.prmtop {name}_gas.rst7
addions protein Na+ 0
addions protein Cl- 0
solvateoct protein OPCBOX {buffer}
""")

# Salt calculation: run tleap once to get water count,
# then append addionsrand and final saveamberparm.
amb.tleap(f=f"{name}.leap.in")

# Parse leap.log for number of water residues
import re
with open("leap.log") as log:
    content = log.read()
match = re.search(r"Added\s+(\d+)\s+residues", content)
if match:
    n_wat = int(match.group(1))
    n_ion_pairs = int(0.0187 * salt_mM * n_wat)
    print(f"Water molecules: {n_wat}, adding {n_ion_pairs} ion pairs")
else:
    n_ion_pairs = 0
    print("Could not detect water count; skipping salt buffer")

# Append neutralization + salt + final save
with open(f"{name}.leap.in", "a") as leap:
    if n_ion_pairs > 0:
        leap.write(f"addionsrand protein Na+ {n_ion_pairs} Cl- {n_ion_pairs}\n")
    leap.write(f"saveamberparm protein {name}_solv.prmtop {name}_solv.rst7\n")
    leap.write("quit\n")

amb.tleap(f=f"{name}.leap.in")

### Minimize — Stage 1 (restrained)

Relax solvent around the restrained protein.

In [ ]:
with open("min1.mdin", "w") as m:
    m.write(f"""&cntrl
  imin=1, ncyc={nmin//2}, maxcyc={nmin}, ntmin=1,
  ntb=1, cut=10.0,
  ntr=1,
  restraint_wt=2.0,
  restraintmask=":@CA,C,N",
  ntpr=50,
/
""")

amb.pmemd(i="min1.mdin", o="min1.mdout", p=f"{name}_solv.prmtop",
          c=f"{name}_solv.rst7", r="min1.rst7", ref=f"{name}_solv.rst7", O=True)

### Minimize — Stage 2 (unrestrained)

Relax the entire system.

In [ ]:
with open("min2.mdin", "w") as m:
    m.write(f"""&cntrl
  imin=1, ncyc={nmin//2}, maxcyc={nmin}, ntmin=1,
  ntb=1, cut=10.0,
  ntr=0,
  ntpr=50,
/
""")

amb.pmemd(i="min2.mdin", o="min2.mdout", p=f"{name}_solv.prmtop",
          c="min1.rst7", r="min2.rst7", O=True)

In [ ]:
# Convert restart to PDB for visualization
amb.ambpdb(p=f"{name}_solv.prmtop", c="min2.rst7", o="min2.pdb")
nv.show_file("min2.pdb")